<a href="https://colab.research.google.com/github/strongman2026/colab/blob/copilot%2Foptimize-comfyui-colab/comfyui_colab_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# cell 1:基础环境与网盘挂载（开机跑一次）
import os, sys, subprocess
from google.colab import drive

print("☁️ 1. 正在挂载 Google Drive (弹窗请点击允许)...")
drive.mount('/content/drive')

COMFY_DIR = "/content/ComfyUI"
FRP_DIR = "/content/frp"
FRPC_BIN = f"{FRP_DIR}/frpc"

# ================= 🚨 必填配置 =================
VPS_IP = "34.153.197.18"  # 你的服务器 IP
FRP_TOKEN = "Kaggle2026!" # 你的服务器密码
# ===============================================

print("📥 2. 部署 ComfyUI 核心与管理器...")
if not os.path.exists(COMFY_DIR):
    os.system(f"git clone https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR} -q")
    os.system(f"git clone https://github.com/ltdrdata/ComfyUI-Manager.git {COMFY_DIR}/custom_nodes/ComfyUI-Manager -q")

print("📦 3. 安装环境依赖...")
if os.path.exists(f"{COMFY_DIR}/requirements.txt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", f"{COMFY_DIR}/requirements.txt", "-q"])

print("🔧 4. 部署 FRP 客户端与配置文件...")
if not os.path.exists(FRPC_BIN):
    os.makedirs(FRP_DIR, exist_ok=True)
    os.system(f"wget -q -O {FRP_DIR}/frp.tar.gz https://github.com/fatedier/frp/releases/download/v0.61.1/frp_0.61.1_linux_amd64.tar.gz")
    os.system(f"tar -xzf {FRP_DIR}/frp.tar.gz -C {FRP_DIR} --strip-components=1")
    os.system(f"chmod +x {FRPC_BIN}")

frpc_config = f"""
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.method = "token"
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_colab"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8188
"""
with open(f"{FRP_DIR}/frpc.toml", "w") as f:
    f.write(frpc_config)

print("✅ Cell 1 完成！基础环境已就绪。")

☁️ 1. 正在挂载 Google Drive (弹窗请点击允许)...
Mounted at /content/drive
📥 2. 部署 ComfyUI 核心与管理器...
📦 3. 安装环境依赖...
🔧 4. 部署 FRP 客户端与配置文件...
✅ Cell 1 完成！基础环境已就绪。


In [3]:
#cell 2
import os

COMFY_DIR = "/kaggle/working/ComfyUI"
MODELS_DIR = f"{COMFY_DIR}/models"
OUTPUT_DIR = f"{COMFY_DIR}/output"

# =================================================================
# 1️⃣ Kaggle 原生数据集（右侧 Data 导入的，0秒挂载，极致读取速度）
# =================================================================
KAGGLE_DATASETS = [
    ("checkpoints", "/kaggle/input/animagine-xl-4-0-comfyui*/*.safetensors"),
    ("checkpoints", "/kaggle/input/datasets/jarredstroman/my-pony-v6-private/*.safetensors"),
    ("loras", "/kaggle/input/datasets/jarredstroman/family-guy-style/*.safetensors"),
]

# =================================================================
# 2️⃣ 外部直链极速下载（用 aria2c 从 C站或 HuggingFace 拉取）
# =================================================================
EXTERNAL_URLS = [
    # 格式: ("文件夹名", "模型直链")
    # ("checkpoints", "https://huggingface.co/.../xxx.safetensors"),
]

print("🧹 1. 清理旧挂载与旧缓存...")
os.system(f"find {MODELS_DIR}/checkpoints -type l -delete")
os.system(f"find {MODELS_DIR}/loras -type l -delete")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("🔗 2. 正在秒级挂载 Kaggle 本地数据集模型...")
for folder, path in KAGGLE_DATASETS:
    os.makedirs(f"{MODELS_DIR}/{folder}", exist_ok=True)
    os.system(f"ln -sf {path} {MODELS_DIR}/{folder}/")
    print(f"  -> 已挂载 [{folder}]: {path.split('/')[-2]}")

if EXTERNAL_URLS:
    print("\n🚀 3. 部署多线程下载神器 aria2...")
    os.system("apt-get -y install -qq aria2")

    print("📥 4. 正在以最高百兆/秒拉取外部直链模型到本地...")
    for folder, url in EXTERNAL_URLS:
        dest_dir = f"{MODELS_DIR}/{folder}"
        os.makedirs(dest_dir, exist_ok=True)
        print(f"⚡ 极限拉取中: {url.split('/')[-1]}")
        # 16线程并发下载，榨干 Kaggle 万兆带宽
        os.system(f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M '{url}' -d '{dest_dir}'")

print("\n🎉 模型部署完毕！全部基于 Kaggle NVMe 高速本地硬盘！")
print(f"🖼️ 你的图片将极速生成至: {OUTPUT_DIR}")

🧹 1. 清理旧挂载与旧缓存...
🔗 2. 正在秒级挂载 Kaggle 本地数据集模型...
  -> 已挂载 [checkpoints]: animagine-xl-4-0-comfyui*
  -> 已挂载 [checkpoints]: my-pony-v6-private
  -> 已挂载 [loras]: family-guy-style

🎉 模型部署完毕！全部基于 Kaggle NVMe 高速本地硬盘！
🖼️ 你的图片将极速生成至: /kaggle/working/ComfyUI/output


In [6]:
# cell 3 下载ae,clip_l,t5xxl_fp16模型
import os
from google.colab import userdata

# 1. 从 Colab Secrets 安全读取 HF Token
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✅ 成功从 Secrets 读取 HF_TOKEN！")
except userdata.SecretNotFoundError:
    print("❌ 找不到 Token！请点击左侧 🔑 图标添加 HF_TOKEN，并允许 Notebook 访问。")

# 2. 安装多线程下载神器 aria2
!apt -y install -qq aria2

# 3. 确保目录存在
!mkdir -p /content/ComfyUI/models/vae/
!mkdir -p /content/ComfyUI/models/clip/

# 4. 极速下载（带上 Token 鉴权头，彻底防拦截）
print("🚀 开始极速下载 VAE (ae.safetensors)...")
!aria2c --header="Authorization: Bearer $HF_TOKEN" --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors -d /content/ComfyUI/models/vae -o ae.safetensors

print("🚀 开始极速下载 CLIP-L (clip_l.safetensors)...")
!aria2c --header="Authorization: Bearer $HF_TOKEN" --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors -d /content/ComfyUI/models/clip -o clip_l.safetensors

print("🚀 开始极速下载满血版 T5XXL (t5xxl_fp16.safetensors，约 9.8GB)...")
!aria2c --header="Authorization: Bearer $HF_TOKEN" --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors -d /content/ComfyUI/models/clip -o t5xxl_fp16.safetensors

print("🎉 所有 Flux 必需的齿轮已带 Token 鉴权安全下载完毕！")

✅ 成功从 Secrets 读取 HF_TOKEN！
aria2 is already the newest version (1.36.0-1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
🚀 开始极速下载 VAE (ae.safetensors)...

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
6d856a|OK  |       0B/s|/content/ComfyUI/models/vae/ae.safetensors

Status Legend:
(OK):download completed.
🚀 开始极速下载 CLIP-L (clip_l.safetensors)...

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
edb686|OK  |       0B/s|/content/ComfyUI/models/clip/clip_l.safetensors

Status Legend:
(OK):download completed.
🚀 开始极速下载满血版 T5XXL (t5xxl_fp16.safetensors，约 9.8GB)...

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
4e872e|OK  |       0B/s|/content/ComfyUI/models/clip/t5xxl_fp16.safetensors

Status Legend:
(OK):download completed.


In [7]:
# cell4 从kaggle中拉取flux1-schnell.safetensors，并放置在两个目录下
import os
from google.colab import userdata

# 1. 从 Colab Secrets 安全读取 Kaggle Token
try:
    os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_API_TOKEN')
    print("✅ 成功从 Colab Secrets 读取 API Token！")
except userdata.SecretNotFoundError:
    print("❌ 找不到 Token！请点击左侧 🔑 图标添加名为 KAGGLE_API_TOKEN 的密钥，并允许 Notebook 访问。")

# 2. 升级 Kaggle CLI 确保支持新版 Token
!pip install -q -U kaggle

# 3. 创建接收目录并下载
!mkdir -p /content/dataset
print("⏳ 正在从 Kaggle 拉取 Flux 模型...")
!kaggle datasets download -d jarredstroman/flux-2d-assets-2026-v12 -p /content/dataset --unzip

# 4. 创建 ComfyUI 模型目录
!mkdir -p /content/ComfyUI/models/unet/
!mkdir -p /content/ComfyUI/models/checkpoints/


# 5. 建立软链接到 ComfyUI 目录
!ln -sf /content/dataset/flux1-schnell.safetensors /content/ComfyUI/models/unet/flux1-schnell.safetensors
!ln -s /content/dataset/flux1-schnell.safetensors /content/ComfyUI/models/checkpoints/flux1-schnell.safetensors

print("✅ 模型已成功挂载到 ComfyUI！")

✅ 成功从 Colab Secrets 读取 API Token！
⏳ 正在从 Kaggle 拉取 Flux 模型...
Dataset URL: https://www.kaggle.com/datasets/jarredstroman/flux-2d-assets-2026-v12
License(s): CC0-1.0
flux-2d-assets-2026-v12.zip: Skipping, found more recently modified local copy (use --force to force download)
ln: failed to create symbolic link '/content/ComfyUI/models/checkpoints/flux1-schnell.safetensors': File exists
✅ 模型已成功挂载到 ComfyUI！


In [22]:
# cell5 最终精简稳定版
#============================================================

import os
import shutil
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

# ----------------------------
# 1) 安装基础依赖
# ----------------------------
os.system("pip install -q -U huggingface_hub")

# ----------------------------
# 2) 安装自定义节点
# ----------------------------
CUSTOM_NODES_DIR = "/content/ComfyUI/custom_nodes"
os.makedirs(CUSTOM_NODES_DIR, exist_ok=True)

plugins = [
    "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",
    "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
    "https://github.com/rgthree/rgthree-comfy.git",
]

for url in plugins:
    repo_name = url.split("/")[-1].replace(".git", "")
    repo_path = os.path.join(CUSTOM_NODES_DIR, repo_name)
    if not os.path.exists(repo_path):
        print(f"⬇️ 正在克隆插件: {repo_name} ...")
        os.system(f"git clone -q {url} {repo_path}")

    req_file = os.path.join(repo_path, "requirements.txt")
    if os.path.exists(req_file):
        print(f"📦 正在安装依赖: {repo_name} ...")
        os.system(f"pip install -q -r {req_file}")

# ----------------------------
# 3) 创建目录
# ----------------------------
for d in [
    "unet", "clip", "vae", "ipadapter", "clip_vision",
    "controlnet", "checkpoints", "loras", "xlabs/ipadapters"
]:
    os.makedirs(f"/content/ComfyUI/models/{d}", exist_ok=True)

os.makedirs("/content/dataset", exist_ok=True)

# ----------------------------
# 4) 安全软链
# ----------------------------
def safe_symlink(src, dst, desc):
    try:
        src = Path(src)
        dst = Path(dst)

        if src.exists() and dst.exists():
            try:
                if src.resolve() == dst.resolve():
                    print(f"ℹ️ {desc} 已在正确位置，跳过：{dst}")
                    return
            except Exception:
                pass

        if dst.is_symlink() or dst.exists():
            if dst.is_dir() and not dst.is_symlink():
                shutil.rmtree(dst)
            else:
                dst.unlink()

        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            os.symlink(str(src), str(dst))
            print(f"✅ {desc}: {src} -> {dst}")
        else:
            print(f"⚠️ 未找到 {desc} 源文件：{src}")
    except Exception as e:
        print(f"❌ {desc} 处理失败：{e}")

# ----------------------------
# 5) Flux 主模型挂载
# ----------------------------
safe_symlink(
    "/content/dataset/flux1-schnell.safetensors",
    "/content/ComfyUI/models/unet/flux1-schnell.safetensors",
    "Flux 主模型 flux1-schnell.safetensors"
)

# ----------------------------
# 6) CLIP Vision 下载/挂载
# ----------------------------
CLIP_VISION_REPO = "shiertier/clip_vision"
CLIP_VISION_FILE = "CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors"
CLIP_VISION_LOCAL = f"/content/dataset/{CLIP_VISION_FILE}"
CLIP_VISION_DST = f"/content/ComfyUI/models/clip_vision/{CLIP_VISION_FILE}"

if not os.path.exists(CLIP_VISION_LOCAL):
    print("⬇️ 开始下载 CLIP Vision ...")
    downloaded = hf_hub_download(
        repo_id=CLIP_VISION_REPO,
        filename=CLIP_VISION_FILE,
        local_dir="/content/dataset"
    )
    print(f"✅ CLIP Vision 下载完成：{downloaded}")

safe_symlink(CLIP_VISION_LOCAL, CLIP_VISION_DST, "CLIP Vision 模型")

# ----------------------------
# 7) 自动发现 Flux IPAdapter 仓库文件并下载
# ----------------------------
api = HfApi()
repo_id = "XLabs-AI/flux-ip-adapter"

print(f"🔎 正在列出仓库文件：{repo_id}")
files = api.list_repo_files(repo_id=repo_id)

safetensor_files = [f for f in files if f.endswith(".safetensors")]

if not safetensor_files:
    print(f"⚠️ 在 {repo_id} 中没有找到 .safetensors 文件")
else:
    chosen_file = None
    for candidate in safetensor_files:
        name = candidate.lower()
        if "adapter" in name:
            chosen_file = candidate
            break
    if chosen_file is None:
        chosen_file = safetensor_files[0]

    print(f"⬇️ 选择下载文件：{chosen_file}")
    local_download = hf_hub_download(
        repo_id=repo_id,
        filename=chosen_file,
        local_dir="/content/dataset"
    )
    print(f"✅ 下载完成：{local_download}")

    FLUX_IPADAPTER_DST = f"/content/ComfyUI/models/xlabs/ipadapters/{Path(chosen_file).name}"
    os.makedirs(os.path.dirname(FLUX_IPADAPTER_DST), exist_ok=True)

    if os.path.exists(FLUX_IPADAPTER_DST) or os.path.islink(FLUX_IPADAPTER_DST):
        try:
            os.remove(FLUX_IPADAPTER_DST)
        except:
            pass

    os.symlink(local_download, FLUX_IPADAPTER_DST)
    print(f"✅ Flux IPAdapter 权重: {local_download} -> {FLUX_IPADAPTER_DST}")

# ----------------------------
# 8) 旧 checkpoints 检查
# ----------------------------
old_ckpt = "/content/ComfyUI/models/checkpoints/flux1-schnell.safetensors"
if os.path.exists(old_ckpt):
    print("⚠️ 检测到旧位置的 Flux 模型：/content/ComfyUI/models/checkpoints/flux1-schnell.safetensors")
    print("   建议后续工作流使用 /content/ComfyUI/models/unet/ 下的模型。")

# ----------------------------
# 9) 输出当前目录状态
# ----------------------------
print("\n📁 当前 /content/ComfyUI/models 文件列表：")
os.system("find /content/ComfyUI/models -maxdepth 3 -type f -o -type l | sort")

print("\n✅ 处理完成。")
print("⚠️ 请重启 ComfyUI，再重新加载工作流。")

📦 正在安装依赖: comfyui_controlnet_aux ...
📦 正在安装依赖: rgthree-comfy ...
ℹ️ Flux 主模型 flux1-schnell.safetensors 已在正确位置，跳过：/content/ComfyUI/models/unet/flux1-schnell.safetensors
ℹ️ CLIP Vision 模型 已在正确位置，跳过：/content/ComfyUI/models/clip_vision/CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors
🔎 正在列出仓库文件：XLabs-AI/flux-ip-adapter
⬇️ 选择下载文件：ip_adapter.safetensors
✅ 下载完成：/content/dataset/ip_adapter.safetensors
✅ Flux IPAdapter 权重: /content/dataset/ip_adapter.safetensors -> /content/ComfyUI/models/xlabs/ipadapters/ip_adapter.safetensors
⚠️ 检测到旧位置的 Flux 模型：/content/ComfyUI/models/checkpoints/flux1-schnell.safetensors
   建议后续工作流使用 /content/ComfyUI/models/unet/ 下的模型。

📁 当前 /content/ComfyUI/models 文件列表：

✅ 处理完成。
⚠️ 请重启 ComfyUI，再重新加载工作流。


In [ ]:
# cell6 跑日常，前面的不需要跑，只需跑一次
import os, time, subprocess, socket

COMFY_DIR = "/content/ComfyUI"
FRP_DIR = "/content/frp"
FRPC_BIN = f"{FRP_DIR}/frpc"
VPS_IP = "34.153.197.18"

# ==================================================
# 🛒 应用商店开关 (ComfyUI-Manager)
# False = 日常极速跑图 (屏蔽 Manager，极速启动)
# True  = 需要安装新插件时打开 (恢复慢速下载缓存)
# ==================================================
ENABLE_MANAGER = False

print("🛑 1. 秒杀旧进程，释放端口...")
os.system("pkill -f 'main.py'")
os.system("pkill -f frpc")
time.sleep(1)

print(f"🎛️ 2. 处理 Manager 状态 (当前: {'开启' if ENABLE_MANAGER else '极致屏蔽'})...")
MANAGER_DIR = f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager"
HIDDEN_MANAGER_DIR = f"{COMFY_DIR}/ComfyUI-Manager_hidden"

if ENABLE_MANAGER:
    if os.path.exists(HIDDEN_MANAGER_DIR):
        os.rename(HIDDEN_MANAGER_DIR, MANAGER_DIR)
else:
    if os.path.exists(MANAGER_DIR):
        os.rename(MANAGER_DIR, HIDDEN_MANAGER_DIR)

print("🚀 3. 极速启动 ComfyUI 主引擎...")
# Colab 免费版分配的通常是 T4，保留 normalvram 防爆显存
comfy = subprocess.Popen(["python3", "main.py", "--normalvram"], cwd=COMFY_DIR)

print("⏳ 4. 等待引擎点火...")
for _ in range(60):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", 8188)) == 0:
            break
    time.sleep(1)

print("🌐 5. 端口就绪！唤醒 FRP 隧道...")
frpc = subprocess.Popen([FRPC_BIN, "-c", f"{FRP_DIR}/frpc.toml"])

print("\n" + "="*50)
print(f"🎉 Colab 引擎已极速拉起！(应用商店: {'已加载' if ENABLE_MANAGER else '已隐藏'})")
print(f"👉 请刷新网页: http://{VPS_IP}:8188")
print("="*50 + "\n")

comfy.wait()